# Olist Data Quality — Notebook educativo

## Fase 1: Preparación del entorno y reconocimiento de las fuentes

**Objetivo:** verificar Python, DuckDB y los archivos de Olist sin alterar CSV. Este notebook será ampliado de acuerdo con el orden del proyecto; **aquí no se implementan aún la extracción completa a RAW ni las 15 reglas**.

Consulta la guía `docs/FASE_1_TUTORIAL.md` para ejecutar los pasos desde Windows 11.

### Arquitectura fija

```text
CSV originales -> Extracción Python + DuckDB -> RAW
              -> 15 reglas SQL -> CLEAN
              -> pruebas críticas -> Parquet + reporte de calidad
```

RAW y CLEAN son las únicas capas persistentes de datos. `audit` será exclusivamente un esquema de control, no una capa STAGING.

### 1. Detectar la carpeta raíz y mostrar el entorno

En VS Code, selecciona **Python (olist-quality)** arriba a la derecha. El bloque siguiente localiza la raíz incluso si el directorio de trabajo es `notebooks/`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in (cwd, *cwd.parents)
                     if (candidate / "environment.yml").exists()), None)
if project_root is None:
    raise RuntimeError("Abre el notebook desde la carpeta olist-data-quality")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Raíz:", project_root)
print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)

### 2. Comprobar bibliotecas

Verificamos las versiones realmente instaladas. Si una importación falla, vuelve al Paso 4 del tutorial.

In [ ]:
import duckdb
import pandas as pd
import pyarrow as pa
import matplotlib
print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Matplotlib:", matplotlib.__version__)

### 3. Crear la base persistente

El módulo `src.bootstrap` es **la única implementación** de esta operación; el notebook sólo lo invoca. Repetir la celda no borra datos.

In [ ]:
from src.bootstrap import bootstrap
from src.config import DB_PATH

print("Base de datos:", bootstrap())
connection = duckdb.connect(str(DB_PATH))
try:
    schema_rows = connection.execute("""
        SELECT schema_name FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'clean', 'audit') ORDER BY schema_name
    """).fetchall()
    print("Esquemas:", schema_rows)
    assert {'raw', 'clean', 'audit'}.issubset({row[0] for row in schema_rows})
finally:
    connection.close()

### 4. Verificar archivos originales

Los nueve archivos oficiales deben copiarse a `data/raw/` sin cambios. Esta celda funciona antes de descargarlos: reporta los faltantes y no inventa cantidades.

In [ ]:
from src.config import DATA_RAW_DIR, EXPECTED_CSV_FILES

present = sorted(path.name for path in DATA_RAW_DIR.glob('*.csv'))
missing = sorted(set(EXPECTED_CSV_FILES) - set(present))
print("CSV encontrados:", len(present))
print("CSV faltantes:", missing)
if missing:
    print("Descarga el dataset Olist siguiendo el Paso 7 del tutorial.")

### 5. Inspeccionar el origen y generar reportes

**Sólo cuando estén los nueve archivos** se ejecuta el inventario completo. Se cuentan registros CSV (no líneas físicas) y se registra una huella SHA-256 que ayuda a verificar inmutabilidad. Los archivos originales no se modifican.

In [ ]:
from src.source_inventory import build_inventory

if missing:
    print("Inventario pendiente: primero descargar los 9 CSV originales.")
else:
    inventory = build_inventory()
    inventory_df = pd.DataFrame(inventory)
    display(inventory_df[["file", "records_count", "columns_count", "row_width_mismatches"]])
    print("Inventario JSON y CSV creado en reports/quality_reports/")

### 6. Comprobar integridad de estructura sin convertir tipos

Los errores de ancho en un CSV se reportan para su investigación. Una fila irregular **no implica automáticamente** que deba eliminarse.

In [ ]:
if missing:
    print("Validación del ancho pendiente: faltan archivos originales.")
else:
    width_issues = inventory_df.loc[inventory_df["row_width_mismatches"] > 0,
                                     ["file", "row_width_mismatches"]]
    if width_issues.empty:
        print("No se observaron filas con cantidad de campos distinta al encabezado.")
    else:
        print("Archivos con anomalías estructurales:")
        display(width_issues)

### 7. Final de Fase 1

Una Fase 1 verificada tendrá: entorno Conda, kernel correcto, `database/ecommerce.duckdb`, nueve archivos Olist en `data/raw/` y reportes de inventario. **Aún no se han creado tablas RAW, CLEAN ni Parquet.**

La Fase 2 usará las columnas reales para determinar PK, FK, claves de negocio, granularidad y relaciones sin suponer que todos los duplicados o nulos son errores.

## Fase 2 — Comprensión de Olist y carga de CSV a RAW

Los CSV originales se preservan. En este capítulo se crean las tablas RAW con todas sus columnas como `VARCHAR`, sin limpieza. El perfilado posterior revela datos faltantes, claves candidatas y relaciones reales; **no se deduplican pedidos, artículos, pagos ni registros geográficos**.

### 8. Claves y granularidad: hipótesis que verificaremos

Una fila en `orders` representa un pedido, en `order_items` representa un artículo de pedido y en `order_payments` un pago. `customer_id` y `customer_unique_id` son diferentes; `review_id` y prefijos postales **no se dan por únicos**. Estas hipótesis del catálogo se contrastan con los datos cargados.

In [ ]:
from src.model_metadata import TABLE_SPECS, RELATIONSHIPS
for spec in TABLE_SPECS:
    print(f"{spec.name:20} | {spec.grain} | clave candidata: {spec.candidate_key or 'ninguna'}")

### 9. Carga RAW transaccional

Python coordina la lectura, DuckDB ejecuta la importación SQL. Se comprueban SHA-256, encabezados, tipos `VARCHAR` y conteos contra el inventario. No se omiten errores de lectura ni se alteran los CSV.

In [ ]:
from src.extract import load_raw
if missing:
    raise FileNotFoundError("No se puede cargar RAW: faltan los CSV indicados en la Fase 1")
raw_load = load_raw()
display(pd.DataFrame(raw_load)[["table", "rows", "columns", "action"]])

### 10. Perfilado inicial, sin modificar RAW

Se generan reportes de columnas, nulos/vacíos, unicidad de claves candidatas, referencias entre tablas, rangos de fechas, valores monetarios y categorías. Los indicadores **son observaciones reales**, no se supone que deban ser cero.

In [ ]:
from src.understand import profile_raw
profile = profile_raw()
for name in ("raw_tables", "raw_keys", "raw_references", "raw_dates", "raw_monetary"):
    print("\n", name)
    display(pd.DataFrame(profile[name]).head(30))

### 11. Porcentaje de datos faltantes por tabla

La figura resume el porcentaje de campos vacíos o nulos detectados en RAW. Un valor ausente no siempre constituye un error de negocio; requiere interpretación contextual.

In [ ]:
import matplotlib.pyplot as plt
missing_df = pd.DataFrame(profile["raw_missingness"])
if not missing_df.empty:
    summary = missing_df.groupby("table", as_index=False)["missing_or_empty_pct"].mean().sort_values("missing_or_empty_pct")
    ax = summary.plot.barh(x="table", y="missing_or_empty_pct", legend=False, figsize=(10, 5))
    ax.set_title("Promedio de porcentaje de valores ausentes por columna en RAW")
    ax.set_xlabel("Porcentaje medio de celdas ausentes")
    ax.set_ylabel("Tabla RAW")
    plt.tight_layout()
    plt.show()

### 12. Verificar conteos CSV vs RAW y revisar la propuesta ER

La cantidad de filas debe coincidir; una diferencia es una falla de la extracción. En el diagrama ER, las claves se señalan como PK **sólo si la prueba de unicidad y completitud pasó**. Las líneas representan relaciones conceptuales, no constraints físicos en RAW.

In [ ]:
source_counts = {row["file"]: row["records_count"] for row in inventory}
comparison = pd.DataFrame(profile["raw_tables"]).copy()
comparison["csv_rows"] = comparison["source"].map(source_counts)
comparison["same_rows"] = comparison["records_count"] == comparison["csv_rows"]
display(comparison[["table", "csv_rows", "records_count", "same_rows"]])
assert comparison["same_rows"].all(), "Los conteos de CSV y RAW difieren"
print("Diagrama ER:", project_root / "reports/data_dictionary/olist_er.mmd")

### 13. Resultado de Fase 2

Hemos inspeccionado los CSV y cargado sus valores en RAW sin convertir ni limpiar. Se crearon archivos de diagnóstico y un diccionario generado a partir de columnas reales. **La siguiente Fase 3** analiza los problemas y prepara el reporte de calidad inicial; **Fase 4** implementará las 15 reglas y construirá CLEAN. No afirmar que existen Parquet o una capa CLEAN procesada hasta ejecutarlo y validarlo.